vamos a ver si podemos usar el rag de azure desde este jupyter notebook que nunca he usado

In [30]:
pip install openai requests python-dotenv azure-search-documents azure-core

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
from dotenv import load_dotenv
load_dotenv()
chat_gpt_api_key = os.getenv("CHAT_GPT_4_API_KEY")
chat_gpt_api_model = os.getenv("CHAT_GPT_4_MODEL")
chat_gpt_api_endpoint = os.getenv("CHAT_GPT_4_ENDPOINT")
chat_gpt_api_temperature = float(os.getenv("CHAT_GPT_4_TEMPERATURE", 0.7))
chat_gpt_api_system_prompt = os.getenv("CHAT_GPT_4_SYSTEM_PROMPT", "You are a helpful assistant.")
chat_gpt_api_version = os.getenv("CHAT_GPT_4_API_VERSION")
index_name = os.getenv("INDEX_NAME")
index_endpoint = os.getenv("INDEX_ENDPOINT")
index_key= os.getenv("INDEX_KEY")
print(f'cargadas las variables de entorno de chat gpt 4')

cargadas las variables de entorno de chat gpt 4


In [32]:
embedding_api_key = os.getenv("EMBEDDING_API_KEY")
embedding_api_endpoint = os.getenv("EMBEDDING_API_ENDPOINT")
embedding_api_model = os.getenv("EMBEDDING_API_MODEL")

print(f'cargadas las variables de entorno de embedding')

cargadas las variables de entorno de embedding


ahora con todo cargado tengo que hacer primero una al vector para tener el valor vectorizado  de la consulta

In [33]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=chat_gpt_api_key,
    api_version="2023-05-15",
    azure_endpoint=chat_gpt_api_endpoint
)



ahora tengo que crear  un nuevo embedding desde el cliente de azure que ha creado antes

In [34]:
def generate_embeddings(client, text):
    response = client.embeddings.create(
        input=text,
        model = "text-embedding-ada-002"
    )
    embeddings=response.model_dump()
    print(f'Embeddings generados {embeddings}', embeddings)
    return embeddings['data'][0]['embedding']

ahora llamo al api para traer el asunto  de la vectorizacion

In [35]:
user_query = input()
vectorised_user_query = generate_embeddings(azure_openai_client, user_query)
print(vectorised_user_query)
context=[]

Embeddings generados {'data': [{'embedding': [-0.029935132712125778, -0.019759390503168106, 0.00804145261645317, -0.014609556645154953, -0.011883174069225788, 0.009618072770535946, -0.02894372120499611, 0.010533751919865608, -0.018024420365691185, -0.026492729783058167, 0.018451279029250145, 0.014802331104874611, -0.007807369343936443, 0.005394245032221079, -0.0035560019314289093, 0.0014544151490554214, 0.009198099374771118, -0.009225639514625072, 0.029549583792686462, -0.005666194949299097, 0.0036455043591558933, 0.016854003071784973, 0.011779901571571827, 0.00782802328467369, -0.006581873632967472, -0.004237597808241844, -0.003106768475845456, -0.028448015451431274, 0.007993258535861969, -0.002814164152368903, 0.018189655616879463, 0.001913975807838142, -0.016909081488847733, 0.005521613638848066, -0.03527774289250374, -0.009239409118890762, -0.001891600200906396, -0.020186249166727066, -0.0042823487892746925, 0.007077579852193594, -0.0004741909506265074, 0.020351484417915344, 0.0124

#vale ya tengo el vectorizado del resultado de la consulta ahora tengo que hacer la consulta al indice y despues al llm

In [36]:
from azure.search.documents.models import VectorizedQuery
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

credential = AzureKeyCredential(index_key)

search_client = SearchClient(endpoint=index_endpoint, index_name=index_name, credential=credential)

vector_query = VectorizedQuery(
    vector=vectorised_user_query,
    k_nearest_neighbors=50,
    fields="text_vector"
)

try:
    # 3. Realizar la Búsqueda Híbrida
    response = search_client.search(
        search_text=user_query,  # <--- PARTE TRADICIONAL (Keyword)
        vector_queries=[vector_query],
        select=["*"],
        top=5
    )
    for doc in response:
        context.append(dict(
        {
            "chunk": doc['chunk'],
            "score": doc['@search.score']

        }
        ))
except Exception as e:
    print('request exception', e)


en los asesinatos), de mayor 
numero que el último que hicieron. Los aldeanos 
no dejaran de luchar hasta que el ultimo de ellos 
muera. 
 
Hombres-Aldeanos (30 o 40): AL N; CA 9; MV 
12; DG 1; pg 4 cada uno; GAC0 20; N°At 1; Dñ 
por arma (garrote o cuchillo); AE no; DE no; RM 
no; TAM M; ML 8; EXP 15 cada uno. 
 
SSeexxttaa  NNoocchhee  
Esta es la ultima noche posible para la Noche de 
Thoth. No hay ataques o eventos en esta noche. 
Los PJ deberían dirigirse esta noche al Valle del 
Descanso del Faraón. El DM deberá conducir a 
los pj hasta aquí; se deja a discreccion del DM dar 
unas cuantas pistas para guiarles. En Muhar, por 
ejemplo, los pj pueden obtener información sobre 
esto; alguno de los aldeanos ha visto dirigirse 
hasta alli a los zombies de Senmet. En cualquier 
caso, el siguiente punto de destino será el Valle 
del Descanso del Faraón. 
 
 
 
 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



TTTTTTTTEEE

In [37]:
system_prompt = f""""Eres un dungeon master que lleva toda la vida jugando a advanced dungeons and dragons e intentas que parezca misterioso el asunto"""

user_prompt = f""" the user query is: {user_query}
the context is : {context}"""

print(user_prompt)

chat_completions_response = azure_openai_client.chat.completions.create(
    model = chat_gpt_api_model,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)
print("***********************")
print(chat_completions_response)
print(chat_completions_response.choices[0].message.content)

 the user query is: eventos del toque de la muerte
the context is : [{'chunk': "en los asesinatos), de mayor \nnumero que el último que hicieron. Los aldeanos \nno dejaran de luchar hasta que el ultimo de ellos \nmuera. \n \nHombres-Aldeanos (30 o 40): AL N; CA 9; MV \n12; DG 1; pg 4 cada uno; GAC0 20; N°At 1; Dñ \npor arma (garrote o cuchillo); AE no; DE no; RM \nno; TAM M; ML 8; EXP 15 cada uno. \n \nSSeexxttaa  NNoocchhee  \nEsta es la ultima noche posible para la Noche de \nThoth. No hay ataques o eventos en esta noche. \nLos PJ deberían dirigirse esta noche al Valle del \nDescanso del Faraón. El DM deberá conducir a \nlos pj hasta aquí; se deja a discreccion del DM dar \nunas cuantas pistas para guiarles. En Muhar, por \nejemplo, los pj pueden obtener información sobre \nesto; alguno de los aldeanos ha visto dirigirse \nhasta alli a los zombies de Senmet. En cualquier \ncaso, el siguiente punto de destino será el Valle \ndel Descanso del Faraón. \n \n \n \n \n \n\n \n \n \n \n \n 